In [1]:
import pandas as pd
#import modin.pandas as pd
#import ray
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(style='whitegrid',font_scale=2)
import os
import pickle
import time

os.chdir(os.getcwd())
os.getcwd()

'/data/gpfs/projects/punim2121/C-Path/outcome_prediction/outcome_prediction'

In [8]:
#ce=pd.read_csv(dir+'/data/ttp_ae.csv', low_memory=False)
#mh=pd.read_csv(dir+'/data/ttp_mh.csv', low_memory=False)
ce=pd.read_csv('../data/out_ce.csv.gz', low_memory=False)
#cm=pd.read_csv(dir+'/data/ttp_cm.csv', low_memory=False)

In [3]:
#load patient IDs who are considered in this  analysis
pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)


<hr>

* ### Drop measurement points with no temporal data

* ### Extract clinical events that originate from way before start of therapy as Medical History



In [4]:
### Initiate standardised column names
ce['STD_CEDY']=ce['CEDY']
ce=ce.drop_duplicates()

#--------------------------------
#### Drop measurement point without any temporal information
ce_with_time=ce[~((ce['CEDY'].isna())&(ce['CESTDY'].isna()))]

### Extract clinical events that originate from way before start of therapy as Medical History
ce_medical_history=ce_with_time[((ce_with_time['CEDY'].isna())&(ce_with_time['CESTDY']<0))|
                                ((ce_with_time['CEDY']<0)|(ce_with_time['CESTDY']<0))|
                                ((ce_with_time['CEDY']<0)&(ce_with_time['CESTDY'].isna()))]

# Extract datapoints that were recorded during therapy (=not considered medical history)
ce_timely_relevant=ce_with_time.loc[~ce_with_time.index.isin(ce_medical_history.index),:]
ce_timely_relevant[(ce_timely_relevant['CEDY'].isna())]['STD_CEDY']=ce_timely_relevant[(ce_timely_relevant['CEDY'].isna())]['CESTDY']
ce_timely_relevant=ce_timely_relevant.dropna(how='all',axis=1)


/var/folders/0_/_m74rlq93855zphvhkg3qkhh19_hwl/T/ipykernel_16460/956911998.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ce_timely_relevant[(ce_timely_relevant['CEDY'].isna())]['STD_CEDY']=ce_timely_relevant[(ce_timely_relevant['CEDY'].isna())]['CESTDY']


In [5]:
ce_timely_relevant['CETOXGR'].value_counts(dropna=False)
#ce_timely_relevant['CESEV'].value_counts(dropna=False)

NaN    208477
<1      91801
1       24916
2       15620
3         586
4          50
Name: CETOXGR, dtype: int64

- ## Standardise occurence of clinical event (yes/no) to ordinal variable (1/0)
- ## Standardise toxicity grades of clinical events

In [5]:
ce_std=ce_timely_relevant.copy()

## Standardise occurence of clinical event (yes/no) to ordinal variable (1/0)
conversion_dict={'Y':1,'N':0}
ce_std['STD_CAT_RESULT']=ce_std['CEOCCUR']
ce_std['STD_CAT_ORDINAL_RESULT']=ce_std['CEOCCUR'].map(conversion_dict)

## Standardise toxicity grades of clinical events
ce_std['STD_CETOXGR']=ce_std['CETOXGR'].replace({r'<1':0},regex=True)

## Where toxicity grtade is missing, impute the grade from the severity information
#  Conversion: mild:1, moderate:2, severe:3, life threatening:4
sev_list=['MILD','MODERATE','SEVERE']
tox_grade_list=[1,2,3]
sev_tox_grade_dict=dict(zip(sev_list,tox_grade_list))
ce_std.loc[ce_std['STD_CETOXGR'].isna(),'STD_CETOXGR']=ce_std.loc[ce_std['STD_CETOXGR'].isna(),'CESEV'].map(sev_tox_grade_dict)
ce_std.loc[~ce_std['STD_CETOXGR'].isna(),'STD_CETOXGR']=ce_std.loc[~ce_std['STD_CETOXGR'].isna(),'STD_CETOXGR'].astype(int) +1

### Drop duplicates and save dataframe

In [6]:
duplicated_rows=ce_std[['STD_CEDY','USUBJID','STD_CETERM']].duplicated()
ce_std=ce_std[~duplicated_rows]
ce_std.to_csv('../data/out_ce_standardised_with_time.csv.gz',compression='gzip')
ce_medical_history.to_csv('../data/out_ce_mh.csv.gz',compression='gzip')

In [8]:
ce_std.shape

(337231, 26)

In [2]:
ce_std=pd.read_csv('../data/out_ce_standardised_with_time.csv.gz',index_col=0)

/tmp/ipykernel_147434/2823185768.py:1: DtypeWarning: Columns (5,7,10,11,14,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  ce_std=pd.read_csv('../data/out_ce_standardised_with_time.csv.gz',index_col=0)


In [11]:
ce_std.loc[(ce_std['USUBJID']=='TB-1021/1084727')\
            &(ce_std['CETERM'].str.contains('SHORTNESS OF BREATH'))\
            ,:]#.value_counts()

,STUDYID,DOMAIN,USUBJID,CESEQ,CEGRPID,CETERM,CECAT,CEPRESP,CEOCCUR,CESEV,...,STSTUDMO,STSTUDYR,AIDCRIT1,AIDCRIT2,AIDCRIT3,STD_CETERM,STD_CEDY,STD_CAT_RESULT,STD_CAT_ORDINAL_RESULT,STD_CETOXGR
16375,TB-1021,CE,TB-1021/1084727,20,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,8.0,N,0.0,NaN
51338,TB-1021,CE,TB-1021/1084727,53,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,36.0,N,0.0,NaN
60845,TB-1021,CE,TB-1021/1084727,61,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,43.0,N,0.0,NaN
71687,TB-1021,CE,TB-1021/1084727,70,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,50.0,N,0.0,NaN
81705,TB-1021,CE,TB-1021/1084727,78,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,57.0,N,0.0,NaN
93118,TB-1021,CE,TB-1021/1084727,87,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,85.0,N,0.0,NaN
103625,TB-1021,CE,TB-1021/1084727,95,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,120.0,N,0.0,NaN
114457,TB-1021,CE,TB-1021/1084727,103,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,155.0,N,0.0,NaN
125694,TB-1021,CE,TB-1021/1084727,111,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,183.0,N,0.0,NaN
136538,TB-1021,CE,TB-1021/1084727,119,NaN,SHORTNESS OF BREATH,NaN,Y,N,NaN,...,NaN,NaN,NaN,NaN,NaN,DYSPNEA,274.0,N,0.0,NaN


In [9]:
ce.loc[(ce['USUBJID']=='TB-1021/1084727')\
            #&(ce_std['CETERM'].str.contains('DYSPNEA'))\
            ,'CETERM'].value_counts()

CETERM
CHEST PAINS                  18
COUGH                        18
FEVER                        18
HEMOPTYSIS                   18
NIGHT SWEATS                 18
SHORTNESS OF BREATH          18
UNINTENTIONAL WEIGHT LOSS    18
Name: count, dtype: int64